In [ ]:
#the below is a test, using the first line from the dataset
lines = open("../data/3I_ATLAS_obs.txt").read().splitlines()
print(len(lines)) #prints the number of lines (starting at 0)

'\u200b0003I         S2025 05 08.51765919 12 35.590-18 42 21.35         21.57VV#0K4XC57'
line0 date:  S2025 05 08.51765
line0 ra:  919 12 35.59
line0 dec:  0-18 42 21.3
line0 obs:  XC5


In [11]:
line0 = lines[0]
print(repr(line0)) #The raw line
print("line0 date: ", line0[15:32]) #the date for this line, with the lower number essentially shifted, as python reads differently than humans
print("line0 ra: ", line0[32:44]) #the right ascencion for this line
print("line0 dec: ", line0[44:56]) #the declination for this line
print("line0 obs: ", line0[77:80]) #info abt the observer for this line

'\u200b0003I         S2025 05 08.51765919 12 35.590-18 42 21.35         21.57VV#0K4XC57'
line0 date:  S2025 05 08.51765
line0 ra:  919 12 35.59
line0 dec:  0-18 42 21.3
line0 obs:  XC5


In [16]:
import pandas as pd

rows = []
skipped = 0 #this creates a counter for lines we couldn't parse
for line0 in lines:
    if len(line0) <80: #checks if the line is shorter tha 80 charachters, and skips if it is broken (which it would be if it was less than 80 charachters)
        skipped += 1
        continue #after the skip counter has been increased, this simply skips any broken lines

    #the below is a fix to make the parser detect if there is a shift from an S at the start of the date, from if the observer was a space telescope. This should fix that error.
    off = 0 if line0[15].isdigit() else 1
    rows.append({ #builds a list for the current line of all of its important info (normalizes data)
        "date_str": line0[15+off:32+off].strip(),
        "ra_str":   line0[32+off:44+off].strip(),
        "dec_str":  line0[44+off:56+off].strip(),
        "obscode":  line0[77+off:80+off],
    })

df = pd.DataFrame(rows) #turns the list into a table
print(len(df), "parsed,", skipped, "skipped") #prints how many rows were skipped, and how many made it to the table
df.head() #shows the first five rows of the table

8433 parsed, 0 skipped


,date_str,ra_str,dec_str,obscode
0,2025 05 08.517659,19 12 35.590,-18 42 21.35,C57
1,2025 05 08.517659,1 -129676.05,3 -109544.63,C57
2,2025 05 10.603515,19 11 27.094,-18 41 21.70,C57
3,2025 05 10.603515,1 -36109.021,8 -211212.99,C57
4,2025 05 12.689361,19 10 11.775,-18 40 49.56,C57


In [18]:
#this cell should convert the date text into readible simestamps
from datetime import datetime, timedelta #uses python's built in date tools. datetime is for a moment in time, and time delta is for an amount of time

def parse_date(s): #creates a program called parse_date that takes a text date and returns a timestamp
    y, m, d = s.split() #splits our text into three peices, year, m onth, and day (fraction of a day)
    day_frac = float(d) #converts the text for the day (ex. 08.51765) into a number
    day = int(day_frac) #keeps the whole number part, and seperates the fraction
    frac = day_frac - day #converts the leftover decimal into the fraction of the day that had passed (ex. 0.41196 to abt 9.9 hrs)
    return datetime(int(y), int(m), day) + timedelta(days=frac) #adds the fraction as actual hours.minutes, returns the completed timestamp

df["time"] = df["date_str"].apply(parse_date) # runs the program on every entry within the date_str column in the dataset, and stores the results in a new column it created, time.
df.head() #prints the timestamps for the first 5 rows of the dataset

,date_str,ra_str,dec_str,obscode,time
0,2025 05 08.517659,19 12 35.590,-18 42 21.35,C57,2025-05-08 12:25:25.737600
1,2025 05 08.517659,1 -129676.05,3 -109544.63,C57,2025-05-08 12:25:25.737600
2,2025 05 10.603515,19 11 27.094,-18 41 21.70,C57,2025-05-10 14:29:03.696000
3,2025 05 10.603515,1 -36109.021,8 -211212.99,C57,2025-05-10 14:29:03.696000
4,2025 05 12.689361,19 10 11.775,-18 40 49.56,C57,2025-05-12 16:32:40.790400


In [20]:
#This cell should administer sanity checks
print("total observations:", len(df)) #len(df) is the number of rows on our table, equilivent to the observations parsed
print("first:", df["time"].min()) #prints out the earliest timesteamp in the time column
print("last: ", df["time"].max()) #prints out the latest timestamp in the time column
print(df["obscode"].value_counts().head(10)) #counts how many times each different observatory code appears, sorted from largest to smallest, and shows the top 10

total observations: 8433
first: 2025-05-08 12:25:25.737600
last:  2026-04-14 23:21:21.801600
obscode
C40    245
C23    234
W68    213
T05    207
213    172
958    140
W84    137
323    122
R17    111
R59    110
Name: count, dtype: int64


In [21]:
#quantifies the entries from TESS
sat = df[df["obscode"] == "C57"]
print(len(sat), sat["time"].min())

30 2025-05-08 12:25:25.737600
